<a href="https://colab.research.google.com/github/Ayushi-369/FlyrankAi-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayushi-369/FlyrankAi-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task Type:** Binary Classification (with Probability Scoring)
**Why:** We are predicting whether a given page is experiencing organic traffic decay and requires a content refresh (`1` = Needs Refresh, `0` = Healthy). Outputting probability scores allows us to rank pages from highest to lowest priority for content rewrites.

In [5]:
import os
import pandas as pd

path = "../../data/raw/content_refresh_anonymized.csv" if os.path.exists("../../data/raw/content_refresh_anonymized.csv") else "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(path)

# Clean column names: strip whitespace and convert to lowercase
df.columns = df.columns.str.strip().str.lower()

# Handle potential NaN values in session columns by filling with 0
df['sessions_last_30d'] = df['sessions_last_30d'].fillna(0)
df['sessions_prev_30d'] = df['sessions_prev_30d'].fillna(0)

# Define 'needs_refresh' as a target column based on traffic decay
# A simple rule: if sessions in the last 30 days dropped compared to the previous 30 days
# This definition can be refined later based on problem requirements.
df['needs_refresh'] = (df['sessions_last_30d'] < df['sessions_prev_30d']).astype(int)

target_col = "needs_refresh"

print("Task Type: Binary Classification")
print(f"Target Distribution:\n{df[target_col].value_counts(normalize=True).round(3)}")

Task Type: Binary Classification
Target Distribution:
needs_refresh
0    0.667
1    0.333
Name: proportion, dtype: float64


## 2. Target or proxy

**Target / Proxy Label:** `needs_refresh` (Binary Flag)
**Label Source:** Defined Rule / Performance Outcome. The label flags pages where 90-day impressions and search position dropped below baseline thresholds without recovering naturally, serving as a proxy for content decay.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Target Column Summary:")
print(df[target_col].value_counts())

Target Column Summary:
needs_refresh
0    20016
1     9984
Name: count, dtype: int64


## 3. Success metric

**Success Metric:** Precision@50 (Top-50 Precision)
**Defense:** Content teams have limited writer capacity (e.g., updating 50 pages/month). They care that the top 50 pages recommended for refresh are truly decaying, avoiding wasted editorial budget. A Precision@50 > 0.65 means at least 33 out of 50 recommended rewrites are high-lift decay targets.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Baseline Precision@50 sorting naively by search volume
top_50_naive = df.sort_values(by="search_volume", ascending=False).head(50)
naive_p50 = top_50_naive[target_col].mean()
print(f"Baseline Precision@50 (Sorting naively by Search Volume): {naive_p50:.3f}")

Baseline Precision@50 (Sorting naively by Search Volume): 0.320


## 4. The unit of analysis, as a real dataframe

**Unit of Analysis:** One published page (URL) evaluated over a trailing 90-day window. One row = One page's performance metrics.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample_cols = [c for c in ["page_id", "content_age_days", "avg_position", "impressions_90d", "ctr", target_col] if c in df.columns]
print("Unit of Analysis DataFrame (1 row = 1 page):")
df[sample_cols].head(5)

Unit of Analysis DataFrame (1 row = 1 page):


,content_age_days,avg_position,impressions_90d,ctr,needs_refresh
0,187,10.6,3803,0.76,1
1,445,20.3,15320,0.05,0
2,141,36.5,12581,0.09,1
3,463,6.2,11751,0.49,0
4,263,44.0,19140,0.13,0


## 5. Why ML beats a fixed rule here

**Why ML Beats Fixed Rules:** Traffic decay depends on complex multi-feature interactions. Simple rules (e.g., "refresh articles older than 365 days") yield low precision because evergreen content stays high-ranking, while newer articles can suffer sudden decay from competitor updates. Machine learning combines position changes, CTR drops, engagement, and age simultaneously to isolate true decay.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Precision of a naive rule: "content_age_days > 365"
old_pages = df[df["content_age_days"] > 365]
rule_precision = old_pages[target_col].mean()
print(f"Precision of simple rule 'content_age_days > 365': {rule_precision:.3f}")
print("Simple age-based rules fail because age alone does not cause decay.")

Precision of simple rule 'content_age_days > 365': 0.270
Simple age-based rules fail because age alone does not cause decay.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.